# 🎯 Measure Utility: Downstream Model Performance

In this notebook we will evaluate whether synthetic data is useful, by measuring downstream model performance and analytical consistency.
Utility asks a different question: *can you actually use it?* A dataset can be statistically similar but still fail to support the models or analyses it was designed for.

**In this notebook, we will:**
1. Evaluate utility via **TSTR** — Train on Synthetic, Test on Real
2. Evaluate utility via **analytical consistency** — do both datasets lead to the same insights?

## 1. Load the Datasets

In [22]:
import pandas as pd
original_df  = pd.read_csv('data/bank-full.csv', sep=';')
synthetic_df = pd.read_csv('data/bank-full_synthetic.csv')

TARGET = 'y'

print(f'Original : {original_df.shape[0]:,} rows')
print(f'Synthetic: {synthetic_df.shape[0]:,} rows')

Original : 45,211 rows
Synthetic: 45,211 rows


In [24]:
original_df = original_df.drop('duration', axis=1) # removing the calculated feature created
synthetic_df = synthetic_df.drop('total_contacts', axis=1) # removing the calculated feature created

## 2. TSTR — Train on Synthetic, Test on Real

TSTR is the most direct utility test for supervised learning: train a model on synthetic data and evaluate it on real data. If the synthetic data captured the right patterns, the model should perform comparably to one trained on real data.

We train the same classifier twice — once on synthetic, once on real — and test both on the same real holdout. The gap between the two F1 scores is the utility cost of using synthetic data.

In [26]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

def encode(df):
    """Label-encode all object columns."""
    df = df.copy()
    for col in df.select_dtypes(include='object').columns:
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))
    return df

# Encode both datasets
orig_enc  = encode(original_df)
synth_enc = encode(synthetic_df)

# Split original into train (80%) and holdout (20%)
# The holdout is the evaluation set for both models
orig_train, holdout = train_test_split(orig_enc, test_size=0.2, random_state=42, stratify=orig_enc[TARGET])

X_holdout = holdout.drop(columns=[TARGET])
y_holdout = holdout[TARGET]

print(f'Holdout size: {len(holdout):,} rows')

Holdout size: 9,043 rows


In [27]:
def train_and_evaluate(X_train, y_train, X_test, y_test, label):
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    return {
        'trained_on': label,
        'accuracy':   round(accuracy_score(y_test, y_pred), 4),
        'f1_score':   round(f1_score(y_test, y_pred, average='weighted'), 4),
    }

# Baseline: trained on real data
result_real  = train_and_evaluate(
    orig_train.drop(columns=[TARGET]), orig_train[TARGET],
    X_holdout, y_holdout,
    label='Real data',
)

# TSTR: trained on synthetic data
result_synth = train_and_evaluate(
    synth_enc.drop(columns=[TARGET]), synth_enc[TARGET],
    X_holdout, y_holdout,
    label='Synthetic data',
)

results_df = pd.DataFrame([result_real, result_synth])
print(results_df.to_string(index=False))

    trained_on  accuracy  f1_score
     Real data    0.8923    0.8672
Synthetic data    0.8999    0.8830


In [28]:
# Utility gap
f1_gap = round(result_real['f1_score'] - result_synth['f1_score'], 4)
print(f'F1 gap (real − synthetic): {f1_gap}')
print('→ ' + ('Strong utility — synthetic data captures the predictive signal.' 
               if abs(f1_gap) < 0.05 else 
               'Notable gap — synthetic data may be missing patterns or relationships.'))

F1 gap (real − synthetic): -0.0158
→ Strong utility — synthetic data captures the predictive signal.


## 3. Analytical Consistency

TSTR tests predictive utility. Analytical consistency tests a different question: if a data analyst ran queries on both datasets, would they reach the same conclusions?

We run three aggregations on both datasets and compare the results side by side.

In [29]:
# Query 1 — Subscription rate by job
def subscription_rate(df):
    return (
        df.groupby('job')['y']
        .apply(lambda x: (x == 'yes').mean())
        .round(3)
        .rename('subscription_rate')
    )

q1 = pd.concat(
    [subscription_rate(original_df).rename('original'),
     subscription_rate(synthetic_df).rename('synthetic')],
    axis=1,
).sort_values('original', ascending=False)

q1['delta'] = (q1['synthetic'] - q1['original']).round(3)
print('=== Subscription rate by job ===')
print(q1.to_string())

=== Subscription rate by job ===
               original  synthetic  delta
job                                      
student           0.287      0.309  0.022
retired           0.228      0.227 -0.001
unemployed        0.155      0.151 -0.004
management        0.138      0.137 -0.001
admin.            0.122      0.123  0.001
self-employed     0.118      0.119  0.001
unknown           0.118      0.170  0.052
technician        0.111      0.117  0.006
services          0.089      0.085 -0.004
housemaid         0.088      0.132  0.044
entrepreneur      0.083      0.113  0.030
blue-collar       0.073      0.082  0.009


In [30]:
# Query 2 — Average balance by education level
def avg_balance(df):
    return (
        df.groupby('education')['balance']
        .mean()
        .round(0)
        .rename('avg_balance')
    )

q2 = pd.concat(
    [avg_balance(original_df).rename('original'),
     avg_balance(synthetic_df).rename('synthetic')],
    axis=1,
)
q2['delta'] = (q2['synthetic'] - q2['original']).round(0)
print('=== Average balance by education ===')
print(q2.to_string())

=== Average balance by education ===
           original  synthetic  delta
education                            
primary      1251.0     1221.0  -30.0
secondary    1155.0     1163.0    8.0
tertiary     1758.0     1739.0  -19.0
unknown      1527.0     1629.0  102.0


In [31]:
# Query 3 — Contact rate by marital status
def avg_contacts(df):
    return (
        df.groupby('marital')['campaign']
        .mean()
        .round(2)
        .rename('avg_contacts')
    )

q3 = pd.concat(
    [avg_contacts(original_df).rename('original'),
     avg_contacts(synthetic_df).rename('synthetic')],
    axis=1,
)
q3['delta'] = (q3['synthetic'] - q3['original']).round(2)
print('=== Average campaign contacts by marital status ===')
print(q3.to_string())

=== Average campaign contacts by marital status ===
          original  synthetic  delta
marital                             
divorced      2.63       2.62  -0.01
married       2.84       2.88   0.04
single        2.65       2.63  -0.02


> **⏸ Active Learning** — Look at the delta column across the three queries. Are the differences large enough to change a business decision? For example, if the subscription rate by job were used to prioritise outreach targets, would an analyst reach the same conclusion using synthetic data? That is the utility question — not whether the numbers match exactly, but whether they lead to the same action.

## ✅ Summary

In this notebook you:

1. **Implemented TSTR** — trained a classifier on synthetic data and tested it on a real holdout, then compared the F1 score against a baseline trained on real data. The gap between the two is the utility cost of using synthetic data.
2. **Measured analytical consistency** — ran three aggregation queries on both datasets and compared the results. Small deltas mean an analyst would reach the same conclusions from either dataset.

The key question is the same in both cases: are these differences acceptable for your use case? A 2% F1 gap may be negligible for exploration but unacceptable for a production model. A delta in subscription rate may or may not change a business decision. Metrics set the threshold; the use case determines whether you've crossed it.